In [1]:
import numpy as np

In [2]:
#Copy the weights from Session 1.
client1_weights = np.array([
    1.36765211, 0.83013565, 0.25391510, 0.49150359,
    0.36293059, 0.19475784, 0.17854927, 0.10473428
])

client2_weights = np.array([
    3.03081649, 0.01040235, -0.74969200, 1.92767140,
    1.59371063, 0.12610740, -0.21419882, 0.78586758
])

print("Client 1:")
print(client1_weights)

print("\nClient 2:")
print(client2_weights)

Client 1:
[1.36765211 0.83013565 0.2539151  0.49150359 0.36293059 0.19475784
 0.17854927 0.10473428]

Client 2:
[ 3.03081649  0.01040235 -0.749692    1.9276714   1.59371063  0.1261074
 -0.21419882  0.78586758]


In [3]:
global_weights = (
    client1_weights +
    client2_weights
) / 2

print("FedAvg Weights:")
print(global_weights)

FedAvg Weights:
[ 2.1992343   0.420269   -0.24788845  1.2095875   0.97832061  0.16043262
 -0.01782478  0.44530093]


### # Reconstructing the VQC from Session 1

In [4]:
import numpy as np

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp

from qiskit_machine_learning.neural_networks import EstimatorQNN

In [5]:
X, y = make_moons(
    n_samples=400,
    noise=0.15,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train:", len(X_train))
print("Test :", len(X_test))

Train: 320
Test : 80


In [6]:
y_train_q = 2 * y_train - 1
y_test_q = 2 * y_test - 1

print(np.unique(y_train_q))

[-1  1]


In [7]:
num_qubits = 4

x_params = ParameterVector("x", 2)
theta = ParameterVector("θ", 8)

qc = QuantumCircuit(num_qubits)

qc.ry(x_params[0], 0)
qc.ry(x_params[1], 1)

for i in range(num_qubits):
    qc.ry(theta[2*i], i)
    qc.rz(theta[2*i + 1], i)

qc.cx(0, 1)
qc.cx(1, 2)
qc.cx(2, 3)

observable = SparsePauliOp.from_list(
    [("ZZZZ", 1)]
)

print(qc)

     ┌──────────┐┌──────────┐┌──────────┐               
q_0: ┤ Ry(x[0]) ├┤ Ry(θ[0]) ├┤ Rz(θ[1]) ├──■────────────
     ├──────────┤├──────────┤├──────────┤┌─┴─┐          
q_1: ┤ Ry(x[1]) ├┤ Ry(θ[2]) ├┤ Rz(θ[3]) ├┤ X ├──■───────
     ├──────────┤├──────────┤└──────────┘└───┘┌─┴─┐     
q_2: ┤ Ry(θ[4]) ├┤ Rz(θ[5]) ├─────────────────┤ X ├──■──
     ├──────────┤├──────────┤                 └───┘┌─┴─┐
q_3: ┤ Ry(θ[6]) ├┤ Rz(θ[7]) ├──────────────────────┤ X ├
     └──────────┘└──────────┘                      └───┘


### Recreate the QNN

In [8]:
from qiskit_machine_learning.neural_networks import EstimatorQNN

qnn = EstimatorQNN(
    circuit=qc,
    input_params=x_params,
    weight_params=theta,
    observables=observable
)

print("QNN created successfully!")

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


QNN created successfully!


In [9]:
sample = X_test[0]

output = qnn.forward(
    input_data=sample,
    weights=global_weights
)

print("Input :", sample)
print("Output:", output)

Input : [-1.04219708  0.23161997]
Output: [[0.99970884]]


# Evaluating the Aggregated Quantum Model

### Predict on Test Set

In [10]:
predictions = []

for sample in X_test:

    output = qnn.forward(
        input_data=sample,
        weights=global_weights
    )

    pred = 1 if output[0][0] >= 0 else -1

    predictions.append(pred)

predictions = np.array(predictions)

print("Predictions created!")
print(predictions[:10])

Predictions created!
[1 1 1 1 1 1 1 1 1 1]


In [11]:
accuracy = np.mean(predictions == y_test_q)

print("FedAvg Test Accuracy:", accuracy)

FedAvg Test Accuracy: 0.475


### Results Cell

In [12]:
print("Client 1 Test Accuracy :", 0.7375)
print("Client 2 Test Accuracy :", 0.7875)
print("FedAvg Test Accuracy   :", accuracy)

Client 1 Test Accuracy : 0.7375
Client 2 Test Accuracy : 0.7875
FedAvg Test Accuracy   : 0.475
